In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import brainmass
import brainstate
import brainunit as u
import jax
import jax.numpy as jnp
import numpy as np
brainstate.environ.set(dt=0.1 * u.ms)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


# Choosing Models

This guide helps you select the appropriate neural mass model for your research question.

## Decision Tree

```text
Need physiological realism?
├─ YES → Physiological Models
│  ├─ Modeling EEG/MEG? → JansenRitStep
│  ├─ Modeling fMRI BOLD? → WongWangStep
│  ├─ E-I population dynamics? → WilsonCowanStep
│  └─ Mean-field spiking? → MontbrioPazoRoxinStep
│
└─ NO → Phenomenological Models
   ├─ Studying oscillations? → HopfStep or StuartLandauStep
   ├─ Studying excitability? → FitzHughNagumoStep
   ├─ Phase synchronization? → KuramotoNetwork
   └─ Fast/simple dynamics? → ThresholdLinearStep
```

## Model Categories

### Phenomenological Models

**When to use:**

- Studying generic dynamical phenomena (bifurcations, synchronization)
- Computational efficiency is critical
- Detailed biophysical mechanisms are not required

**Hopf Oscillator**

- **Best for:** Onset of oscillations, rhythm generation
- **Key feature:** Supercritical Hopf bifurcation
- **Variables:** 2 (x, y)
- **Example use:** Studying how oscillatory activity emerges

In [2]:
# Construct a Hopf oscillator population (90 regions)
hopf = brainmass.HopfStep(
    in_size=90,
    w=0.1,  # intrinsic frequency (dimensionless)
    a=0.1,  # >0 for a stable limit cycle
)
hopf.init_all_states()

# Advance the population for a few hundred steps with brainstate's for_loop
# (never drive a model with a bare Python loop).
def hopf_step(i):
    hopf.update()
    return hopf.x.value

hopf_trace = brainstate.transform.for_loop(hopf_step, jnp.arange(500))
print("Hopf trace shape:", hopf_trace.shape)  # (n_steps, n_regions)

Hopf trace shape: (500, 90)


**FitzHugh-Nagumo**

- **Best for:** Excitable systems, spike generation
- **Key feature:** Type II excitable, fast-slow dynamics
- **Variables:** 2 (v, w)
- **Example use:** Studying transitions from rest to spiking

**Kuramoto Network**

- **Best for:** Phase synchronization in oscillator networks
- **Key feature:** Order parameter for collective synchrony
- **Variables:** 1 (θ)
- **Example use:** Studying synchronization patterns

### Physiological Models

**When to use:**

- Linking models to empirical neuroimaging data
- Biophysical realism is important
- Modeling specific neural populations

**Jansen-Rit Model**

- **Best for:** EEG signal generation, alpha rhythms
- **Key feature:** 3 neural populations (pyramidal, excitatory, inhibitory)
- **Variables:** 6 states
- **Example use:** Simulating EEG from cortical columns

In [3]:
# Jansen-Rit population for 68 cortical sources (6 state variables each)
jr = brainmass.JansenRitStep(
    in_size=68,  # number of cortical sources
)
jr.init_all_states()
print("Jansen-Rit in_size:", jr.in_size)

Jansen-Rit in_size: (68,)


**Wilson-Cowan Model**

- **Best for:** Excitatory-inhibitory population dynamics
- **Key feature:** Classic E-I model, firing rate equations
- **Variables:** 2 (rE, rI)
- **Example use:** Studying E-I balance, oscillations

**Wong-Wang Model**

- **Best for:** Resting-state fMRI, decision-making tasks
- **Key feature:** Reduced spiking network with NMDA/GABA synapses
- **Variables:** 2 (S_E, S_I)
- **Example use:** Whole-brain resting-state fMRI simulations

**Montbrio-Pazo-Roxin Model**

- **Best for:** Mean-field reduction of spiking networks
- **Key feature:** Exact mean-field reduction of quadratic integrate-and-fire neuron networks
- **Variables:** 2 (r, v)
- **Example use:** Linking spiking and rate dynamics

## Model Comparison

| Model | Complexity | Speed | Realism | Typical Application |
| --- | --- | --- | --- | --- |
| ThresholdLinear | Low | Very Fast | Low | Fast exploratory simulations |
| Hopf | Low | Fast | Low | Oscillation emergence |
| FitzHugh-Nagumo | Low | Fast | Medium | Excitability, spikes |
| Kuramoto | Low | Fast | Low | Phase synchronization |
| Wilson-Cowan | Medium | Medium | Medium | E-I dynamics, general purpose |
| Jansen-Rit | High | Slow | High | EEG/MEG modeling |
| Wong-Wang | High | Slow | High | fMRI BOLD, decision making |
| Montbrio-Pazo-Roxin | Medium | Medium | High | Mean-field spiking networks |

## Use Case Examples

### Resting-State fMRI Study

**Goal:** Simulate spontaneous BOLD fluctuations matching empirical FC

**Recommended:** {class}`WongWangStep` or {class}`WilsonCowanStep`

**Why:**
- Slow synaptic dynamics (NMDA) match BOLD timescales
- Captures E-I dynamics generating realistic fluctuations
- Well-validated for resting-state simulations

In [4]:
# Wong-Wang nodes (AAL90 atlas) feeding a Balloon-Windkessel BOLD model
nodes = brainmass.WongWangStep(in_size=90)  # AAL90 atlas
bold = brainmass.BOLDSignal(in_size=90)
nodes.init_all_states()
bold.init_all_states()

# Structural coupling from a DTI connectome would be added here via a
# coupling object (see the adding_coupling tutorial).
print("Wong-Wang + BOLD nodes ready:", nodes.in_size, bold.in_size)

Wong-Wang + BOLD nodes ready: (90,) (90,)


### EEG Source Modeling

**Goal:** Simulate EEG signals from cortical sources

**Recommended:** {class}`JansenRitStep`

**Why:**

- Explicitly models pyramidal neuron membrane potentials (EEG source)
- Validated against empirical EEG spectra
- Generates realistic alpha rhythms

In [5]:
# Jansen-Rit cortical sources projected to scalp electrodes via a lead-field
jr = brainmass.JansenRitStep(in_size=68)  # cortical parcels
jr.init_all_states()

n_regions, n_sensors = 68, 64
# In practice the lead-field comes from a forward solver (e.g. via MNE).
# Here we build a synthetic one with the expected unit (sensor / dipole).
leadfield_matrix = (
    jnp.asarray(np.random.RandomState(0).rand(n_regions, n_sensors))
    * (u.mV / (u.nA * u.meter))
)

eeg_fwd = brainmass.EEGLeadFieldModel(
    in_size=68,
    out_size=64,  # electrodes
    L=leadfield_matrix,
)
print("EEG forward model:", eeg_fwd.in_size, "->", eeg_fwd.out_size)

EEG forward model: (68,) -> (64,)


### Synchronization Study

**Goal:** Study emergence of network synchronization

**Recommended:** {class}`HopfStep` or {class}`KuramotoNetwork`

**Why:**

- Simple, interpretable dynamics
- Well-studied analytically
- Fast for large networks

In [6]:
# Kuramoto for phase-only synchronization.
# omega holds the (dimensionless) intrinsic frequencies; the model converts
# the drive to a rate internally, so omega is a plain array, not a Quantity.
rng = np.random.RandomState(1)
omega = jnp.asarray(0.1 + 0.02 * rng.randn(100))  # heterogeneous frequencies
kuramoto = brainmass.KuramotoNetwork(
    in_size=100,
    omega=omega,
    K=1.0,  # global coupling strength
)
kuramoto.init_all_states()

def kuramoto_step(i):
    kuramoto.update()
    return kuramoto.theta.value

phases = brainstate.transform.for_loop(kuramoto_step, jnp.arange(500))
order = jnp.abs(jnp.mean(jnp.exp(1j * phases), axis=1))  # R in [0, 1]
print("Kuramoto order parameter (final):", float(order[-1]))

# Hopf for amplitude + phase
hopf = brainmass.HopfStep(in_size=100, w=0.2, a=0.1)
hopf.init_all_states()

Kuramoto order parameter (final): 0.9998431205749512


NewStateCatcher(
  state_to_exclude=Nothing(),
  state_tag=None,
  state_ids={137180193202416, 137180193202848},
  states=[
    HiddenState(
      value=ShapedArray(float32[100], weak_type=True)
    ),
    HiddenState(
      value=ShapedArray(float32[100], weak_type=True)
    )
  ]
)

### Parameter Fitting Study

**Goal:** Fit model parameters to empirical data

**Recommended:** Start simple, then increase complexity

**Strategy:**

1. Start with {class}`HopfStep` or {class}`WilsonCowanStep` (fewer parameters)
2. Validate parameter estimates
3. If needed, move to {class}`JansenRitStep` or {class}`WongWangStep`

**Why:**

- Simpler models have fewer local minima
- Easier to interpret fitted parameters
- Can always add complexity later

## Common Pitfalls

**Using Complex Models Too Early**

- More parameters = harder optimization
- Start simple, add complexity only if needed

**Ignoring Model Timescales**

- Jansen-Rit: τ ~ 10 ms → good for EEG (ms resolution)
- Wong-Wang: τ ~ 100 ms → good for fMRI (second resolution)
- Match model timescale to data modality

**Not Considering Computational Cost**

- Jansen-Rit: 6 variables per region → 6× slower than Hopf
- For exploratory work, use faster models

**Mismatched Observable**

- EEG: Use models with explicit membrane potentials (Jansen-Rit)
- fMRI BOLD: Use models with slow synaptic dynamics (Wong-Wang)
- MEG: Similar to EEG

## Mixing Models

You can use different models for different regions:

In [7]:
# Thalamus: fast dynamics
N_thal = 10
thalamus = brainmass.HopfStep(in_size=N_thal, w=0.3)

# Cortex: slower E-I dynamics
N_cort = 80
cortex = brainmass.WilsonCowanStep(in_size=N_cort)

thalamus.init_all_states()
cortex.init_all_states()

# Project the thalamic output onto the cortex
W_thal_cort = jnp.ones((N_cort, N_thal)) * 0.1  # thalamus -> cortex

def network_step(i):
    thalamus.update()  # advance the thalamus
    thal_drive = (W_thal_cort @ thalamus.x.value).mean()  # thalamic drive
    cort_out = cortex.update(rE_inp=thal_drive)
    return cort_out

# Drive the coupled system with for_loop, not a bare Python loop
cort_trace = brainstate.transform.for_loop(network_step, jnp.arange(300))
print("Coupled cortex trace shape:", cort_trace.shape)

Coupled cortex trace shape: (300, 80)


## Model Extensions

All models support:

- **Noise:** Add stochastic fluctuations
- **Coupling:** Connect in networks
- **Batching:** Run multiple parameter sets in parallel
- **Optimization:** Fit parameters with gradient descent

See corresponding tutorials for details.

## Summary Recommendations

| Your Goal | Recommended Model |
| --- | --- |
| Learn brainmass basics | {class}`HopfStep` or {class}`WilsonCowanStep` |
| EEG/MEG modeling | {class}`JansenRitStep` |
| fMRI BOLD modeling | {class}`WongWangStep` or {class}`WilsonCowanStep` |
| Fast exploratory work | {class}`Hopf Oscillator` or {class}`ThresholdLinearStep` |
| Excitable dynamics | {class}`FitzHughNagumoStep` |
| Spiking network reduction | {class}`MontbrioPazoRoxinStep` |
| Synchronization studies | {class}`KuramotoNetwork` or {class}`HopfStep` |

## Next Steps

- {doc}`building_networks` - Create multi-region networks with your chosen model
- {doc}`../apis/models` - Detailed documentation for each model
- {doc}`../examples/index` - See models in action

## See Also

- Deco et al. (2008). "The Dynamic Brain" for model selection guidance
- {doc}`parameter_fitting` for fitting model parameters